# VibeThinker-3B Inference

**Run after model access/download is configured.**

This notebook demonstrates how to load and run inference with VibeThinker-3B
using the Hugging Face `transformers` library.

---

### Prerequisites

- Python 3.10+
- `transformers>=4.54.0` installed
- CUDA-capable GPU with 8GB+ VRAM (for bfloat16 inference)
- ~6GB free disk space for model weights

### Attribution

Model: VibeThinker-3B by **WeiboAI and contributors**

- Original GitHub: https://github.com/WeiboAI/VibeThinker
- Original HF: https://huggingface.co/WeiboAI/VibeThinker-3B
- Mirror HF: https://huggingface.co/OMCHOKSI108/VibeThinker-3B

---

## 1. Install Dependencies

Run this cell once to install required packages.

In [ ]:
# !pip install transformers>=4.54.0 torch
# !pip install vllm==0.10.1  # optional, for faster inference

## 2. Load Model and Tokenizer

Choose either the original WeiboAI repo or the OMCHOKSI108 mirror.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "WeiboAI/VibeThinker-3B"
# Alternative mirror:
# MODEL_NAME = "OMCHOKSI108/VibeThinker-3B"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=True,
    torch_dtype="bfloat16",
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)
print(f"Model loaded: {MODEL_NAME}")

## 3. Inference

Run a prompt through the model. The recommended parameters are based on the
original technical report.

In [ ]:
from transformers import GenerationConfig

prompt = "What is the sum of the first 100 prime numbers?"

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    generation_config=GenerationConfig(
        max_new_tokens=40960,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=None,
    ),
)
response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
)
print(response)

## 4. Custom Prompt

Edit the `prompt` variable below with your own input.

In [ ]:
prompt = "Write a Python function to check if a string is a palindrome."

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    generation_config=GenerationConfig(
        max_new_tokens=40960,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=None,
    ),
)
response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
)
print(response)

## 5. Using vLLM (Optional)

If you have vLLM installed, this cell shows the equivalent inference code.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(MODEL_NAME, dtype="bfloat16", tensor_parallel_size=1)
params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    top_k=-1,
    max_tokens=40960,
)
outputs = llm.generate(["What is 23 * 47?"], params)
print(outputs[0].outputs[0].text)

---

## Notes

- First run downloads ~6GB of model weights from Hugging Face.
- Inference on CPU is possible but significantly slower.
- The model is designed for verifiable reasoning tasks (math, code, STEM).
- See the [original README](../ORIGINAL_README.md) for usage guidelines and recommendations.
- See [docs/INFERENCE.md](../docs/INFERENCE.md) for detailed parameter explanations.